# Example Use Case: Banking Assistant
create a banking customer support assistant. This example will show how to:
- Structure training data for domain-specific knowledge
- Fine-tune a model for specialized tasks
- Integrate the fine-tuned model into a practical application

In [ ]:
%pip install --upgrade openai langchain langgraph==0.3.1 tiktoken pandas scikit-learn

import os, json, time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

client = OpenAI()

# Example Format
```jsonl
{
  "messages": [
    {"role": "system", "content": "You are a helpful domain expert assistant."},
    {"role": "user", "content": "What is the policy on this matter?"},
    {"role": "assistant", "content": "According to our policies..."}
  ]
}
```

### Creating Training Data: A Banking Example
To demonstrate these principles, let's create a training dataset using a banking knowledge base. This example can be adapted for any domain-specific knowledge:

In [ ]:
import json
import random
import pathlib
import pandas as pd


# Create directory for our fine-tuning data
DATA_DIR = pathlib.Path("bank_finetune_data")
DATA_DIR.mkdir(exist_ok=True)


# Example using banking domain
kb_docs = [
    {
        "title": "Account Types",
        "content": {
            "checking": "We offer three types of checking accounts: Basic (no minimum balance), Premium ($2,500 minimum, no fees), and Student (no fees with valid student ID).",
            "savings": "Our savings accounts include Regular (0.5% APY), High-Yield (1.5% APY with $10,000 minimum), and Goal-Based savings with customizable targets.",
            "business": "Business accounts feature unlimited transactions, merchant services integration, and dedicated support. Available in Standard and Premium tiers."
        }
    },
]

def generate_diverse_examples(kb_doc):
    """
    Generate multiple training examples with diverse phrasings and scenarios.
    Following best practices for high-quality training data.
    """
    examples = []
    content = kb_doc["content"]
    
    # Generate examples for each subtopic
    for subtopic, details in content.items():
        questions = [
            f"Can you explain {subtopic}?",
            f"What should I know about {subtopic}?",
            f"Tell me about your {subtopic}",
            f"How does {subtopic} work?",
            f"I need information regarding {subtopic}",
            f"What are the details of {subtopic}?"
        ]
        
        responses = [
            f"Here's what you need to know about {subtopic}: {details}",
            f"Regarding {subtopic}: {details} Let me know if you need any clarification.",
            f"I'll explain {subtopic}. {details} Is there anything specific you'd like to know more about?",
            f"{details} This information about {subtopic} is current as of today. Please check our website for any updates."
        ]
        
        # Generate combinations with system message variations
        system_messages = [
            "You are a knowledgeable banking assistant focused on providing accurate, compliant information.",
            "You are a helpful financial services expert committed to clear, precise communication.",
            "You are a banking specialist dedicated to providing detailed, accurate responses."
        ]
        
        # Create multiple examples for each subtopic
        for system_msg in system_messages:
            for question in questions:
                for response in responses:
                    examples.append({
                        "messages": [
                            {"role": "system", "content": system_msg},
                            {"role": "user", "content": question},
                            {"role": "assistant", "content": response}
                        ]
                    })
    
    return examples

# Generate comprehensive training dataset
examples = []
for doc in kb_docs:
    examples.extend(generate_diverse_examples(doc))

# Shuffle examples to ensure good distribution
random.shuffle(examples)

print(f"Generated {len(examples)} diverse training examples")


### Split into training and validation sets

A crucial step in fine-tuning is dividing your dataset into training and validation sets. The training set teaches the model, while the validation set helps measure its performance on unseen data. We'll use a 80/20 split - a common ratio that provides enough validation data while maximizing training examples.

In [ ]:

random.shuffle(examples)

split_index = int(0.8 * len(examples))

train_examples, validation_examples = examples[:split_index], examples[split_index:]

train_file = DATA_DIR / "train.jsonl"
validation_file = DATA_DIR / "validation.jsonl"

with open(train_file, "w", encoding="utf-8") as f:
    for example in train_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

with open(validation_file, "w", encoding="utf-8") as f:
    for example in validation_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print(f"✍️ Created {len(train_examples)} training examples and {len(validation_examples)} validation examples")

# Create a DataFrame for easy visualization of example structure
df = pd.DataFrame([
    {
        "system": ex["messages"][0]["content"],
        "user": ex["messages"][1]["content"],
        "assistant": ex["messages"][2]["content"]
    }
    for ex in train_examples[:3]
])
# Display first 3 examples showing system, user, and assistant messages
df

In [ ]:
def validate_jsonl(file_path):
    """Validate that the file is proper JSONL format for fine-tuning."""
    with open(file_path, 'r', encoding='utf-8') as f:
        line_count = 0
        for line in f:
            line_count += 1
            try:
                data = json.loads(line)
                # Check if it has the messages field
                if 'messages' not in data:
                    return False, f"Line {line_count} missing 'messages' field"
                # Check message format
                for msg in data['messages']:
                    if 'role' not in msg or 'content' not in msg:
                        return False, f"Line {line_count} has message missing 'role' or 'content'"
                    if msg['role'] not in ['system', 'user', 'assistant']:
                        return False, f"Line {line_count} has invalid role: {msg['role']}"
            except json.JSONDecodeError:
                return False, f"Line {line_count} is not valid JSON"
    return True, f"Validated {line_count} examples"

# Validate our files
train_valid, train_msg = validate_jsonl(train_file)
val_valid, val_msg = validate_jsonl(validation_file)

print(f"Training file: {'✅' if train_valid else '❌'} {train_msg}")
print(f"Validation file: {'✅' if val_valid else '❌'} {val_msg}")

In [ ]:
import openai
import os
from pathlib import Path

# Initialize OpenAI client
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def upload(path):
    """Upload a file to OpenAI for fine-tuning"""
    with open(path, "rb") as file:
        resp = client.files.create(
            file=file,
            purpose="fine-tune"  # Specify the file will be used for fine-tuning
        )
    print(f"Uploaded {Path(path).name} → {resp.id}")
    return resp.id

# Upload our training and validation files
# These IDs will be used when creating the fine-tuning job
train_file_id = upload(train_file)
validation_file_id = upload(validation_file)

In [ ]:
job = client.fine_tuning.jobs.create(
    # The ID of the JSONL file containing training examples
    training_file=train_file_id,
    # The ID of the JSONL file containing validation examples (optional but recommended)
    validation_file=validation_file_id,
    model="gpt-4o-mini-2024-07-18",  # base model
    suffix="banking-support"  # custom suffix for the fine-tuned model
)
# Print job information for monitoring
print(f"Fine-tuning job created with ID: {job.id}")
print(f"Job status: {job.status}")
print(f"Monitor progress at: https://platform.openai.com/finetune")

In [ ]:
import time

# Continuously check job status every 30 seconds
while True:
    # Retrieve current status of the fine-tuning job
    status = client.fine_tuning.jobs.retrieve(job.id).status
    print("Status:", status)
    if status in ("succeeded", "failed", "cancelled"):
        break      # Exit loop if job reaches a terminal state
    # Wait 30 seconds before next status check
    time.sleep(30)

# Once complete, get the identifier for the fine-tuned model
fine_tuned_model = openai.fine_tuning.jobs.retrieve(job.id).fine_tuned_model  # e.g. 'ft:gpt‑4o-mini:banking-support:2025-05-12-10-30-02'
print("✅ Fine‑tuned model:", fine_tuned_model)

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent


# Define example tool to demonstrate fine-tuned model integration
@tool
def account_lookup(account_id: str) -> str:
    """Look up account information by account ID (requires authentication)."""
    return f"Account {account_id} information is available after authentication."

@tool
def transfer_payment(source_account: str, destination_account: str, amount: float) -> str:
    """Transfer payment from one account to another."""
    return f"Payment of {amount} from {source_account} to {destination_account} has been initiated."


# Use our fine-tuned model
domain_expert = ChatOpenAI(model=fine_tuned_model, temperature=0.0)

# Create the agent with our tools
agent = create_react_agent(
    model=domain_expert,  # Use our fine-tuned model
    tools=[account_lookup, transfer_payment]
    )

# Test the agent with some example queries
test_queries = [
    "What account types do you offer?",
    "What are the fees for wire transfers?",
    "I need information about overdraft protection"
]

# Run the agent on a test query
for query in test_queries:
    response = agent.invoke({"messages": [HumanMessage(content=query)]})
    print("\nQuery:", query)
    print("\nResponse:")
    # Extract and print only the final assistant response
    for msg in reversed(response["messages"]):
        # Check if it's an AIMessage and has non-empty content
        if hasattr(msg, "content") and msg.content and msg.__class__.__name__ == "AIMessage":
            print(msg.content)
            break